# Reward-Modulated Spiking Neural Network with Homeostasis

## Architecture Overview
- Fully connected pool of neurons with dedicated inhibitory neurons
- **Separate input neurons**: Pattern inputs (neurons 0-3), Reward/Pain inputs (neurons 4-5)
- **Output neurons**: Read from pool neurons 6-9
- **Homeostatic mechanisms**: Synaptic scaling, adaptive threshold, weight decay
- Reward increases weights, pain decreases weights when exceeding threshold

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import json

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

## Network Configuration with Separate Input Channels

In [ ]:
# Network architecture
N_TOTAL = 50                    # Total neurons in the pool
N_EXCITATORY = 40               # Excitatory neurons (80%)
N_INHIBITORY = 10               # Inhibitory neurons (20%)

# Neuron type indices
EXCITATORY_IDS = list(range(0, 40))     # Neurons 0-23
INHIBITORY_IDS = list(range(40, 50))    # Neurons 24-29

# SEPARATE input neuron groups
PATTERN_INPUT_IDS = [0, 1, 2, 3, 5, 6, 7        # Data pattern inputs (excitatory)
REWARD_INPUT_ID = [8]                    # Reward signal input (excitatory)
PAIN_INPUT_ID = [9]                      # Pain signal input (excitatory)

# Output neurons (read from pool)
OUTPUT_IDS = [10,11, 12]               # Output read from these excitatory neurons

# Neuron dynamics
V_THRESHOLD_EXC = 1.0            # Firing threshold for excitatory neurons
V_THRESHOLD_INH = 0.7            # Lower threshold for inhibitory (fire more easily)
V_RESET = 0.0

# Input strengths
PATTERN_INPUT_STRENGTH = 0.2     # How strongly pattern data drives neurons
REWARD_INPUT_STRENGTH = 0.4      # How strongly reward signal enters
PAIN_INPUT_STRENGTH = 0.4        # How strongly pain signal enters

# Learning parameters
LEARNING_RATE = 0.01
ELIGIBILITY_DECAY = 0.95         # Slower decay for longer credit assignment
PAIN_THRESHOLD = 0.5             # Pain must exceed this to weaken weights
WEIGHT_MAX = 5.0
WEIGHT_MIN_EXC = 0.0             # Excitatory weights stay positive
WEIGHT_MIN_INH = -5.0            # Inhibitory weights stay negative

# HOMEOSTATIC MECHANISMS
TARGET_FIRING_RATE = 0.1         # Target: each neuron fires 10% of steps
SYNAPTIC_SCALING_INTERVAL = 100  # Apply scaling every N steps
ADAPTIVE_THRESHOLD_RISE = 0.3    # How much threshold increases after spike
ADAPTIVE_THRESHOLD_DECAY = 0.95  # Decay factor for adaptive threshold
WEIGHT_DECAY_RATE = 0.999        # Gentle weight decay (forgetting)

print(f"Network: {N_TOTAL} neurons ({N_EXCITATORY} excitatory, {N_INHIBITORY} inhibitory)")
print(f"Pattern inputs: {PATTERN_INPUT_IDS}")
print(f"Reward input: {REWARD_INPUT_ID}")
print(f"Pain input: {PAIN_INPUT_ID}")
print(f"Output neurons: {OUTPUT_IDS}")

## Initialize Network with Neuron Types and Homeostasis

In [ ]:
class SpikingNeuralNetwork:
    """
    Fully connected spiking neural network with:
    - Separate input channels for data, reward, and pain
    - Dedicated inhibitory neurons
    - Homeostatic mechanisms (synaptic scaling, adaptive threshold, weight decay)
    """
    
    def __init__(self):
        self.n_total = N_TOTAL
        
        # Create neuron type masks
        self.is_excitatory = tf.constant(
            [1.0 if i in EXCITATORY_IDS else 0.0 for i in range(N_TOTAL)],
            dtype=tf.float32
        )
        self.is_inhibitory = tf.constant(
            [1.0 if i in INHIBITORY_IDS else 0.0 for i in range(N_TOTAL)],
            dtype=tf.float32
        )
        
        # Base thresholds differ by neuron type
        self.base_threshold = tf.constant(
            [V_THRESHOLD_EXC if i in EXCITATORY_IDS else V_THRESHOLD_INH 
             for i in range(N_TOTAL)],
            dtype=tf.float32
        )
        
        # Initialize weight matrix (all-to-all connectivity)
        initializer = tf.keras.initializers.GlorotUniform()
        W_raw = initializer(shape=(N_TOTAL, N_TOTAL))
        
        # Apply type constraints to initial weights
        W_init = self._apply_weight_constraints(W_raw)
        self.W = tf.Variable(W_init, dtype=tf.float32, name='weights')
        
        # Membrane potentials
        self.V = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32, name='potential')
        
        # Adaptive thresholds (starts at base threshold)
        self.adaptive_threshold = tf.Variable(
            self.base_threshold, 
            dtype=tf.float32, 
            name='adaptive_threshold'
        )
        
        # Spike states
        self.spikes = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32, name='spikes')
        self.prev_spikes = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32, name='prev_spikes')
        
        # Eligibility traces for credit assignment
        self.eligibility = tf.Variable(
            tf.zeros((N_TOTAL, N_TOTAL)), 
            dtype=tf.float32, 
            name='eligibility'
        )
        
        # Bias (small, different for excitatory vs inhibitory)
        bias_init = tf.concat([
            tf.random.uniform((N_EXCITATORY,), -0.01, 0.01),
            tf.random.uniform((N_INHIBITORY,), 0.02, 0.05)  # Slightly positive bias for inhibitory
        ], axis=0)
        self.bias = tf.Variable(bias_init, dtype=tf.float32, name='bias')
        
        # Homeostasis: spike counter for synaptic scaling
        self.spike_counter = tf.Variable(tf.zeros(N_TOTAL), dtype=tf.float32)
        self.steps_since_scaling = tf.Variable(0, dtype=tf.int32)
        
        # Statistics
        self.total_reward = tf.Variable(0.0)
        self.total_pain = tf.Variable(0.0)
        
    def _apply_weight_constraints(self, W):
        """Ensure excitatory neurons send positive weights, inhibitory send negative"""
        # For each column j (pre-synaptic neuron)
        for j in range(N_TOTAL):
            if j in EXCITATORY_IDS:
                # Excitatory pre-synaptic: weights must be >= 0
                col = W[:, j]
                W = tf.tensor_scatter_nd_update(
                    W, 
                    tf.stack([tf.range(N_TOTAL), tf.fill([N_TOTAL], j)], axis=1),
                    tf.maximum(col, 0.0)
                )
            elif j in INHIBITORY_IDS:
                # Inhibitory pre-synaptic: weights must be <= 0
                col = W[:, j]
                W = tf.tensor_scatter_nd_update(
                    W,
                    tf.stack([tf.range(N_TOTAL), tf.fill([N_TOTAL], j)], axis=1),
                    tf.minimum(col, 0.0)
                )
        
        # Remove self-connections
        W = tf.linalg.set_diag(W, tf.zeros(N_TOTAL))
        
        return W
    
    def reset_state(self):
        """Reset network activity but keep learned weights"""
        self.V.assign(tf.zeros(N_TOTAL))
        self.adaptive_threshold.assign(self.base_threshold)
        self.spikes.assign(tf.zeros(N_TOTAL))
        self.prev_spikes.assign(tf.zeros(N_TOTAL))
        self.eligibility.assign(tf.zeros((N_TOTAL, N_TOTAL)))
        self.spike_counter.assign(tf.zeros(N_TOTAL))
        self.steps_since_scaling.assign(0)

# Create network
network = SpikingNeuralNetwork()

print("Network initialized!")
print(f"Total connections: {N_TOTAL * (N_TOTAL - 1)}")
print(f"Excitatory neurons: {N_EXCITATORY}")
print(f"Inhibitory neurons: {N_INHIBITORY}")
print(f"Adaptive thresholds range: [{tf.reduce_min(network.adaptive_threshold):.2f}, {tf.reduce_max(network.adaptive_threshold):.2f}]")

## Create External Inputs (Separate Channels)

**Key design**: Pattern data, reward, and pain each go to DIFFERENT input neurons

In [ ]:
@tf.function
def create_external_input(pattern_values, reward_signal=0.0, pain_signal=0.0):
    """
    Create input vector with SEPARATE channels:
    - Pattern data -> neurons 0-3
    - Reward signal -> neuron 4
    - Pain signal -> neuron 5
    """
    ext = tf.zeros(N_TOTAL, dtype=tf.float32)
    
    # 1. Pattern data goes to dedicated pattern input neurons
    pattern_indices = tf.constant(PATTERN_INPUT_IDS)
    pattern_updates = tf.cast(pattern_values, tf.float32) * PATTERN_INPUT_STRENGTH
    ext = tf.tensor_scatter_nd_update(
        ext, 
        tf.expand_dims(pattern_indices, 1), 
        pattern_updates
    )
    
    # 2. Reward goes to its OWN input neuron (separate from pattern)
    if reward_signal > 0:
        reward_index = tf.constant(REWARD_INPUT_ID)
        reward_update = tf.ones(1) * reward_signal * REWARD_INPUT_STRENGTH
        ext = tf.tensor_scatter_nd_update(
            ext,
            tf.expand_dims(reward_index, 1),
            reward_update
        )
    
    # 3. Pain goes to its OWN input neuron (separate from pattern and reward)
    if pain_signal > 0:
        pain_index = tf.constant(PAIN_INPUT_ID)
        pain_update = tf.ones(1) * pain_signal * PAIN_INPUT_STRENGTH
        ext = tf.tensor_scatter_nd_update(
            ext,
            tf.expand_dims(pain_index, 1),
            pain_update
        )
    
    return ext

# Test the separate channels
test_ext = create_external_input(
    pattern_values=[1.0, 0.5, 0.3, 0.1],
    reward_signal=0.8,
    pain_signal=0.3
)

print("Input channels test:")
print(f"  Pattern neurons (0-3): {test_ext.numpy()[:4]}")
print(f"  Reward neuron (4):    {test_ext.numpy()[4]:.3f}")
print(f"  Pain neuron (5):      {test_ext.numpy()[5]:.3f}")
print(f"  Other neurons (6-9):  {test_ext.numpy()[6:10]}")
print("\n✓ Pattern, reward, and pain go to DIFFERENT neurons!")

## Neuron Dynamics with Adaptive Threshold

In [ ]:
@tf.function
def update_neurons(V, W, bias, prev_spikes, external_input, 
                   adaptive_threshold, base_threshold):
    """
    Update neurons with ADAPTIVE THRESHOLD (homeostasis).
    
    Neurons that fire a lot get higher thresholds (harder to fire).
    This prevents runaway excitation naturally.
    """
    # Sum all weighted inputs
    weighted_sum = tf.linalg.matvec(W, prev_spikes)
    
    # Update potentials
    V_new = V + weighted_sum + bias + external_input
    
    # Check spikes using ADAPTIVE threshold (not fixed)
    spikes_new = tf.cast(V_new >= adaptive_threshold, tf.float32)
    
    # Reset neurons that fired
    V_new = tf.where(spikes_new > 0, V_RESET, V_new)
    
    # Update adaptive threshold:
    # 1. Decay toward base threshold
    # 2. Increase for neurons that just spiked
    threshold_new = base_threshold + \
                    (adaptive_threshold - base_threshold) * ADAPTIVE_THRESHOLD_DECAY + \
                    spikes_new * ADAPTIVE_THRESHOLD_RISE
    
    return V_new, spikes_new, threshold_new

print("Neuron dynamics with adaptive threshold ready!")
print(f"- Base threshold: excitatory={V_THRESHOLD_EXC}, inhibitory={V_THRESHOLD_INH}")
print(f"- After spike: threshold += {ADAPTIVE_THRESHOLD_RISE}")
print(f"- Decay factor: {ADAPTIVE_THRESHOLD_DECAY} per step")

## Weight Modification with Type Constraints

In [ ]:
@tf.function
def modify_weights(W, eligibility, reward_signal, pain_signal):
    """
    Modify weights based on reward and pain.
    Then apply type constraints (excitatory stay positive, inhibitory stay negative).
    """
    weight_change = tf.zeros_like(W)
    
    # Reward: strengthen active connections
    if reward_signal > 0:
        weight_change += LEARNING_RATE * reward_signal * eligibility
    
    # Pain: weaken active connections (only above threshold)
    if pain_signal > PAIN_THRESHOLD:
        pain_strength = pain_signal - PAIN_THRESHOLD
        weight_change -= LEARNING_RATE * pain_strength * eligibility * 2.0
    
    # Apply changes
    W_new = W + weight_change
    
    # Apply type constraints:
    # For each pre-synaptic neuron j:
    for j in range(N_TOTAL):
        if j in EXCITATORY_IDS:
            # Excitatory sources: keep weights >= 0
            col = W_new[:, j]
            W_new = tf.tensor_scatter_nd_update(
                W_new,
                tf.stack([tf.range(N_TOTAL), tf.fill([N_TOTAL], j)], axis=1),
                tf.maximum(col, WEIGHT_MIN_EXC)
            )
        elif j in INHIBITORY_IDS:
            # Inhibitory sources: keep weights <= 0
            col = W_new[:, j]
            W_new = tf.tensor_scatter_nd_update(
                W_new,
                tf.stack([tf.range(N_TOTAL), tf.fill([N_TOTAL], j)], axis=1),
                tf.minimum(col, 0.0)
            )
    
    # Clip all weights to valid range
    W_new = tf.clip_by_value(W_new, WEIGHT_MIN_INH, WEIGHT_MAX)
    
    # Remove self-connections
    W_new = tf.linalg.set_diag(W_new, tf.zeros(N_TOTAL))
    
    return W_new, weight_change

print("Weight modification with type constraints ready!")
print("- Excitatory neurons: weights always ≥ 0")
print("- Inhibitory neurons: weights always ≤ 0")

## Homeostatic Mechanisms

In [ ]:
@tf.function
def synaptic_scaling(W, spike_counts, target_rate=TARGET_FIRING_RATE):
    """
    HOMEOSTASIS: Synaptic Scaling
    
    Scale ALL incoming weights to each neuron to maintain target firing rate.
    - Neurons firing too much -> scale DOWN their inputs
    - Neurons firing too little -> scale UP their inputs
    """
    total_steps = tf.cast(SYNAPTIC_SCALING_INTERVAL, tf.float32)
    
    # Calculate actual firing rate
    actual_rate = spike_counts / total_steps
    
    # Scaling factor: target / actual (clipped to prevent extreme changes)
    scaling = target_rate / (actual_rate + 1e-6)
    scaling = tf.clip_by_value(scaling, 0.1, 10.0)
    
    # Scale each neuron's INCOMING weights (rows of W)
    W_scaled = W * tf.reshape(scaling, (-1, 1))
    
    # Remove self-connections
    W_scaled = tf.linalg.set_diag(W_scaled, tf.zeros(N_TOTAL))
    
    return W_scaled


@tf.function
def weight_decay(W, decay_rate=WEIGHT_DECAY_RATE):
    """
    HOMEOSTASIS: Weight Decay
    
    All weights slowly decay toward zero.
    Only actively used and rewarded connections survive.
    This is "use it or lose it" - prevents old patterns from dominating.
    """
    W_decayed = W * decay_rate
    W_decayed = tf.linalg.set_diag(W_decayed, tf.zeros(N_TOTAL))
    return W_decayed


print("Homeostatic mechanisms ready!")
print(f"1. Synaptic scaling: every {SYNAPTIC_SCALING_INTERVAL} steps, target rate={TARGET_FIRING_RATE}")
print(f"2. Adaptive threshold: rises {ADAPTIVE_THRESHOLD_RISE} per spike, decays {ADAPTIVE_THRESHOLD_DECAY}/step")
print(f"3. Weight decay: {WEIGHT_DECAY_RATE} per step (gentle forgetting)")

## Read Output from Pool

In [ ]:
def read_output(spikes, output_ids=OUTPUT_IDS):
    """Read output spikes from designated output neurons"""
    return tf.gather(spikes, output_ids)

def read_potentials(V, output_ids=OUTPUT_IDS):
    """Read membrane potentials from output neurons"""
    return tf.gather(V, output_ids)

print(f"Output read from neurons: {OUTPUT_IDS}")
print("Same mechanism as input - just reading spike states")

## Complete Step Function with All Mechanisms

In [ ]:
def step(network, pattern_values, reward_signal=0.0, pain_signal=0.0):
    """
    Execute one complete step with ALL mechanisms:
    1. Separate input channels for pattern, reward, pain
    2. Neuron dynamics with adaptive threshold
    3. Eligibility traces
    4. Weight modification from reward/pain
    5. Weight decay (gentle forgetting)
    6. Synaptic scaling (periodic)
    """
    # 1. Create input with SEPARATE channels
    ext_input = create_external_input(pattern_values, reward_signal, pain_signal)
    
    # 2. Update neurons with adaptive threshold
    V_new, spikes_new, threshold_new = update_neurons(
        network.V, network.W, network.bias, network.prev_spikes, ext_input,
        network.adaptive_threshold, network.base_threshold
    )
    
    # 3. Update eligibility traces
    spike_pairs = tf.tensordot(
        tf.expand_dims(spikes_new, 1),
        tf.expand_dims(network.prev_spikes, 0),
        axes=1
    )
    eligibility_new = ELIGIBILITY_DECAY * network.eligibility + spike_pairs
    
    # 4. Modify weights based on reward/pain (with type constraints)
    W_new, weight_changes = modify_weights(
        network.W, eligibility_new, reward_signal, pain_signal
    )
    
    # 5. HOMEOSTASIS: Apply weight decay (gentle forgetting)
    W_new = weight_decay(W_new)
    
    # 6. Read output
    output_spikes = read_output(spikes_new)
    output_potentials = read_potentials(V_new)
    
    # Update network state
    network.V.assign(V_new)
    network.spikes.assign(spikes_new)
    network.prev_spikes.assign(spikes_new)
    network.eligibility.assign(eligibility_new)
    network.W.assign(W_new)
    network.adaptive_threshold.assign(threshold_new)
    
    # Track spikes for synaptic scaling
    network.spike_counter.assign_add(spikes_new)
    network.steps_since_scaling.assign_add(1)
    
    # 7. HOMEOSTASIS: Apply synaptic scaling periodically
    if network.steps_since_scaling >= SYNAPTIC_SCALING_INTERVAL:
        W_scaled = synaptic_scaling(network.W, network.spike_counter)
        network.W.assign(W_scaled)
        network.spike_counter.assign(tf.zeros(N_TOTAL))
        network.steps_since_scaling.assign(0)
    
    # Track statistics
    network.total_reward.assign_add(reward_signal)
    network.total_pain.assign_add(pain_signal)
    
    return output_spikes, output_potentials, weight_changes

print("Complete step function ready!")
print("Each step includes: separate inputs → adaptive neurons → learning → homeostasis")

## Run Simulation with Reward/Pain Cycles

In [ ]:
import gymnasium as gym

env = gym.make(
    "LunarLanderContinuous-v3",
    render_mode="human"
)

obs, info = env.reset()

done = False

while not done:
    action = env.action_space.sample()

    obs, reward, terminated, truncated, info = env.step(action)

    print("Reward:", reward, "act",action)

    done = terminated or truncated

env.close()

In [18]:
# Create fresh network
net = SpikingNeuralNetwork()

env = gym.make(
    "LunarLanderContinuous-v3",
    render_mode="human"
)

obs, info = env.reset()

done = False
# Simulation parameters
N_STEPS = 800
pattern_data = obs.tolist()  #[0.8, 0.4, 0.2, 0.1]  # Constant pattern input

# History tracking
spike_history = []
potential_history = []
threshold_history = []
weight_history = []
output_history = []
reward_pain_history = []
weight_change_history = []

print("Starting simulation with homeostatic mechanisms...")
print(f"Pattern inputs: neurons {PATTERN_INPUT_IDS}")
print(f"Reward input: neuron {REWARD_INPUT_ID}")
print(f"Pain input: neuron {PAIN_INPUT_ID}")
print()
action = env.action_space.sample()
for t in range(N_STEPS):

    obs, reward, terminated, truncated, info = env.step(action)

    reward_sig , pain_sig = 0.0,0.0
    
    if reward > 0:
        reward_sig = reward
    else:
        pain_sig = reward* -1

        
    # Run step
    out_spikes, out_potentials, w_changes = step(
        net, pattern_data, reward_sig, pain_sig
    )
    left = out_spikes[1]
    right = out_spikes[2]

    left * -1
    action = left + right 
    # Store history
    spike_history.append(net.spikes.numpy().copy())
    potential_history.append(net.V.numpy().copy())
    threshold_history.append(net.adaptive_threshold.numpy().copy())
    weight_history.append(net.W.numpy().copy())
    output_history.append(out_spikes.numpy().copy())
    reward_pain_history.append((reward_sig, pain_sig))
    weight_change_history.append(tf.reduce_sum(tf.abs(w_changes)).numpy())
    
    if t % 100 == 0:
        active = int(tf.reduce_sum(net.spikes).numpy())
        exc_active = int(tf.reduce_sum(net.spikes[:24]).numpy())
        inh_active = int(tf.reduce_sum(net.spikes[24:]).numpy())
        print(f"Step {t}: Active: {active} (E:{exc_active} I:{inh_active}), Output: {out_spikes.numpy()}")

print("\n✓ Simulation complete!")
print(f"Total reward received: {net.total_reward.numpy():.1f}")
print(f"Total pain received: {net.total_pain.numpy():.1f}")
print(f"Final avg threshold (E): {tf.reduce_mean(net.adaptive_threshold[:24]).numpy():.2f}")
print(f"Final avg threshold (I): {tf.reduce_mean(net.adaptive_threshold[24:]).numpy():.2f}")

NameError: name 'SpikingNeuralNetwork' is not defined

In [ ]:
# Create fresh network
net = SpikingNeuralNetwork()

# Simulation parameters
N_STEPS = 800
pattern_data = [0.8, 0.4, 0.2, 0.1]  # Constant pattern input

# History tracking
spike_history = []
potential_history = []
threshold_history = []
weight_history = []
output_history = []
reward_pain_history = []
weight_change_history = []

print("Starting simulation with homeostatic mechanisms...")
print(f"Pattern inputs: neurons {PATTERN_INPUT_IDS}")
print(f"Reward input: neuron {REWARD_INPUT_ID}")
print(f"Pain input: neuron {PAIN_INPUT_ID}")
print()

for t in range(N_STEPS):
    # Create reward/pain schedule
    cycle = t % 200
    
    if cycle < 80:
        # Reward period
        reward_sig = 0.8
        pain_sig = 0.0
    elif cycle < 120:
        # Neutral (mild pain, below threshold)
        reward_sig = 0.0
        pain_sig = 0.3
    elif cycle < 180:
        # Pain period (above threshold - weakens weights)
        reward_sig = 0.0
        pain_sig = 0.9
    else:
        # Recovery (no signals)
        reward_sig = 0.0
        pain_sig = 0.0
    
    # Run step
    out_spikes, out_potentials, w_changes = step(
        net, pattern_data, reward_sig, pain_sig
    )
    
    # Store history
    spike_history.append(net.spikes.numpy().copy())
    potential_history.append(net.V.numpy().copy())
    threshold_history.append(net.adaptive_threshold.numpy().copy())
    weight_history.append(net.W.numpy().copy())
    output_history.append(out_spikes.numpy().copy())
    reward_pain_history.append((reward_sig, pain_sig))
    weight_change_history.append(tf.reduce_sum(tf.abs(w_changes)).numpy())
    
    if t % 100 == 0:
        active = int(tf.reduce_sum(net.spikes).numpy())
        exc_active = int(tf.reduce_sum(net.spikes[:24]).numpy())
        inh_active = int(tf.reduce_sum(net.spikes[24:]).numpy())
        print(f"Step {t}: Active: {active} (E:{exc_active} I:{inh_active}), Output: {out_spikes.numpy()}")

print("\n✓ Simulation complete!")
print(f"Total reward received: {net.total_reward.numpy():.1f}")
print(f"Total pain received: {net.total_pain.numpy():.1f}")
print(f"Final avg threshold (E): {tf.reduce_mean(net.adaptive_threshold[:24]).numpy():.2f}")
print(f"Final avg threshold (I): {tf.reduce_mean(net.adaptive_threshold[24:]).numpy():.2f}")

## Visualize Network Activity

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(18, 14))

# 1. Spike raster (with neuron type markers)
ax = axes[0, 0]
spike_array = np.array(spike_history)
ax.imshow(spike_array.T, aspect='auto', cmap='binary', interpolation='nearest')
ax.axhline(y=EXCITATORY_IDS[-1]+0.5, color='blue', linestyle='-', linewidth=2, alpha=0.7)
ax.text(N_STEPS/2, EXCITATORY_IDS[-1]/2, 'EXCITATORY', ha='center', va='center', color='blue', fontweight='bold')
ax.text(N_STEPS/2, EXCITATORY_IDS[-1] + N_INHIBITORY/2, 'INHIBITORY', ha='center', va='center', color='red', fontweight='bold')
ax.set_xlabel('Time Step')
ax.set_ylabel('Neuron ID')
ax.set_title('Spike Raster (Blue=Excitatory, Red=Inhibitory)')

# 2. Reward/Pain input signals (SEPARATE from pattern)
ax = axes[0, 1]
rp_array = np.array(reward_pain_history)
ax.fill_between(range(N_STEPS), 0, rp_array[:, 0], alpha=0.5, color='green', label='Reward Input (neuron 4)')
ax.fill_between(range(N_STEPS), 0, -rp_array[:, 1], alpha=0.5, color='red', label='Pain Input (neuron 5)')
ax.axhline(y=PAIN_THRESHOLD, color='red', linestyle='--', alpha=0.5, label=f'Pain Threshold ({PAIN_THRESHOLD})')
ax.set_xlabel('Time Step')
ax.set_ylabel('Signal Strength')
ax.set_title('Reward & Pain Inputs (Separate Channels)')
ax.legend()

# 3. Adaptive thresholds over time
ax = axes[0, 2]
threshold_array = np.array(threshold_history)
sample_neurons = [0, 5, 10, 15, 24, 27]  # Mix of E and I
for i in sample_neurons:
    label = f'N{i} ({"E" if i in EXCITATORY_IDS else "I"})'
    color = 'blue' if i in EXCITATORY_IDS else 'red'
    ax.plot(threshold_array[:, i], label=label, color=color, alpha=0.7)
ax.set_xlabel('Time Step')
ax.set_ylabel('Threshold')
ax.set_title('Adaptive Thresholds (Homeostasis)')
ax.legend(fontsize=8)

# 4. Excitatory vs Inhibitory activity
ax = axes[1, 0]
exc_active = np.sum(spike_array[:, EXCITATORY_IDS], axis=1)
inh_active = np.sum(spike_array[:, INHIBITORY_IDS], axis=1)
ax.plot(exc_active, label='Excitatory Active', color='blue', alpha=0.7)
ax.plot(inh_active, label='Inhibitory Active', color='red', alpha=0.7)
ax.set_xlabel('Time Step')
ax.set_ylabel('Active Neurons')
ax.set_title('Excitatory vs Inhibitory Activity')
ax.legend()

# 5. Output neuron activity
ax = axes[1, 1]
output_array = np.array(output_history)
for i in range(len(OUTPUT_IDS)):
    ax.plot(output_array[:, i], alpha=0.7, label=f'Output {OUTPUT_IDS[i]}')
ax.set_xlabel('Time Step')
ax.set_ylabel('Spike')
ax.set_title('Output Neuron Activity')
ax.legend()

# 6. Weight changes over time
ax = axes[1, 2]
ax.plot(weight_change_history)
ax.set_xlabel('Time Step')
ax.set_ylabel('Total Weight Change')
ax.set_title('Weight Modification Magnitude')

# 7. Initial weight matrix
ax = axes[2, 0]
im = ax.imshow(weight_history[0], cmap='RdBu', aspect='auto', vmin=-2, vmax=2)
ax.set_xlabel('From (Pre)')
ax.set_ylabel('To (Post)')
ax.set_title('Initial Weight Matrix')
plt.colorbar(im, ax=ax)

# 8. Final weight matrix
ax = axes[2, 1]
im = ax.imshow(weight_history[-1], cmap='RdBu', aspect='auto', vmin=-2, vmax=2)
ax.set_xlabel('From (Pre)')
ax.set_ylabel('To (Post)')
ax.set_title('Final Weight Matrix')
plt.colorbar(im, ax=ax)

# 9. Weight distribution change
ax = axes[2, 2]
init_flat = weight_history[0].flatten()
final_flat = weight_history[-1].flatten()
ax.hist(init_flat[init_flat != 0], bins=30, alpha=0.5, label='Initial', color='blue')
ax.hist(final_flat[final_flat != 0], bins=30, alpha=0.5, label='Final', color='orange')
ax.set_xlabel('Weight Value')
ax.set_ylabel('Count')
ax.set_title('Weight Distribution')
ax.legend()

plt.suptitle('Spiking Network with Homeostasis and Separate Input Channels', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Analyze Homeostatic Effects

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Firing rate distribution (should cluster around target rate)
ax = axes[0]
firing_rates = np.mean(np.array(spike_history), axis=0)
ax.bar(range(N_TOTAL), firing_rates, color=['blue' if i in EXCITATORY_IDS else 'red' for i in range(N_TOTAL)])
ax.axhline(y=TARGET_FIRING_RATE, color='green', linestyle='--', linewidth=2, label=f'Target rate ({TARGET_FIRING_RATE})')
ax.set_xlabel('Neuron ID')
ax.set_ylabel('Firing Rate')
ax.set_title('Firing Rates (Synaptic Scaling Target)')
ax.legend()

# 2. Threshold evolution (should stabilize)
ax = axes[1]
threshold_array = np.array(threshold_history)
avg_exc_threshold = np.mean(threshold_array[:, EXCITATORY_IDS], axis=1)
avg_inh_threshold = np.mean(threshold_array[:, INHIBITORY_IDS], axis=1)
ax.plot(avg_exc_threshold, label='Avg Excitatory Threshold', color='blue')
ax.plot(avg_inh_threshold, label='Avg Inhibitory Threshold', color='red')
ax.set_xlabel('Time Step')
ax.set_ylabel('Average Threshold')
ax.set_title('Adaptive Threshold Stabilization')
ax.legend()

# 3. Weight stability (should not explode)
ax = axes[2]
for i in range(N_TOTAL):
    incoming_weights = [w[i, :].sum() for w in weight_history[::10]]  # Sample every 10 steps
    ax.plot(range(0, N_STEPS, 10), incoming_weights, alpha=0.3, linewidth=0.5)
ax.set_xlabel('Time Step')
ax.set_ylabel('Sum of Incoming Weights')
ax.set_title('Weight Stability (No Explosion)')

plt.tight_layout()
plt.show()

print("Homeostasis analysis:")
print(f"Mean firing rate: {np.mean(firing_rates):.3f} (target: {TARGET_FIRING_RATE})")
print(f"Std firing rate: {np.std(firing_rates):.3f}")
print(f"Weight range final: [{np.min(final_flat):.3f}, {np.max(final_flat):.3f}]")

## Test Separated Input Channels

In [ ]:
def test_input_separation():
    """Verify that pattern, reward, and pain go to different neurons"""
    test_net = SpikingNeuralNetwork()
    
    # Test 1: Only pattern input
    ext1 = create_external_input([1.0, 1.0, 1.0, 1.0], 0.0, 0.0)
    pattern_active = tf.reduce_sum(ext1[PATTERN_INPUT_IDS])
    reward_active = ext1[REWARD_INPUT_ID[0]]
    pain_active = ext1[PAIN_INPUT_ID[0]]
    
    print("Test 1 - Only pattern input:")
    print(f"  Pattern neurons active: {pattern_active.numpy():.2f}")
    print(f"  Reward neuron active:  {reward_active.numpy():.2f}")
    print(f"  Pain neuron active:    {pain_active.numpy():.2f}")
    
    # Test 2: Pattern + Reward
    ext2 = create_external_input([1.0, 1.0, 1.0, 1.0], 0.8, 0.0)
    print("\nTest 2 - Pattern + Reward:")
    print(f"  Pattern neurons: {ext2[PATTERN_INPUT_IDS].numpy()}")
    print(f"  Reward neuron:   {ext2[REWARD_INPUT_ID[0]].numpy():.3f}")
    print(f"  Pain neuron:     {ext2[PAIN_INPUT_ID[0]].numpy():.3f}")
    
    # Test 3: Pattern + Pain
    ext3 = create_external_input([1.0, 1.0, 1.0, 1.0], 0.0, 0.9)
    print("\nTest 3 - Pattern + Pain:")
    print(f"  Pattern neurons: {ext3[PATTERN_INPUT_IDS].numpy()}")
    print(f"  Reward neuron:   {ext3[REWARD_INPUT_ID[0]].numpy():.3f}")
    print(f"  Pain neuron:     {ext3[PAIN_INPUT_ID[0]].numpy():.3f}")
    
    print("\n✓ Confirmed: Pattern, reward, and pain go to DIFFERENT neurons!")

test_input_separation()

## Summary of Improvements

### 1. Separate Input Channels
- **Pattern data** → neurons 0-3 (sensory input)
- **Reward signal** → neuron 4 (neuromodulator)
- **Pain signal** → neuron 5 (neuromodulator)
- No more conflating "what you see" with "how you feel about it"

### 2. Dedicated Inhibitory Neurons
- **Excitatory (neurons 0-23)**: Only send positive weights
- **Inhibitory (neurons 24-29)**: Only send negative weights
- Creates natural braking system, prevents runaway excitation

### 3. Homeostatic Mechanisms
- **Adaptive threshold**: Neurons that fire a lot become harder to fire
- **Synaptic scaling**: Maintains target firing rate across all neurons
- **Weight decay**: Gentle forgetting of unused connections

### 4. Pain Mechanism
- Pain enters through its OWN input neuron (separate from data)
- When pain > threshold → directly weakens recently active connections
- No separate training algorithm needed